In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import gpytorch
import torch
import matplotlib.pyplot as plt
# with pd.ExcelFile("datasets/stresstool_update.xls") as xls:
#     df0 = pd.read_excel(xls, "UserInterface")
#     df1 = pd.read_excel(xls, "Kriging")
#     df2 = pd.read_excel(xls, "Quadratic")

In [15]:
df_data = pd.read_csv("datasets/Kriging_data.csv", header=[0, 1])
df_data.columns

MultiIndex([(        't_lf', 'Unnamed: 0_level_1'),
            (    't_solder', 'Unnamed: 1_level_1'),
            ('MidDieStress',               'top2'),
            ('MidDieStress',               'top5'),
            ('MidDieStress',               'bot4'),
            ('MidDieStress',               'bot5'),
            ('MaxDieStress',               'top2'),
            ('MaxDieStress',               'top5'),
            ('MaxDieStress',               'bot4'),
            ('MaxDieStress',               'bot5')],
           )

In [16]:
X_bounds = np.array([[.2, 1.], [.02, .08]])

In [17]:
X = df_data[["t_lf", "t_solder"]].values

In [18]:
X_scaled = torch.tensor((X - X_bounds[:, 0]) / (X_bounds[:, 1] - X_bounds[:, 0]))

In [19]:
k = 1
N = len(X_scaled)
# for i in range(N // k):
#     plt.scatter(X_scaled[:, 0], X_scaled[:, 1])
#     plt.scatter(X_scaled[k * i:k * (i + 1), 0], X_scaled[k * i:k * (i + 1), 1])
#     plt.xlim(-.05, 1.05)
#     plt.ylim(-.05, 1.05)
#     plt.show()

In [20]:
y = np.abs(df_data[('MidDieStress', 'top2')].values[:, None])

In [21]:
scaler = StandardScaler()
y_scaled = torch.tensor(scaler.fit_transform(y))

In [22]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

def train(X_scaled, y_scaled):
    # initialize likelihood and model
    likelihood = gpytorch.likelihoods.GaussianLikelihood(noise_constraint=gpytorch.constraints.Interval(1e-4, 1e-3))
    # likelihood = gpytorch.likelihoods.GaussianLikelihood()
    model = ExactGPModel(X_scaled, y_scaled.flatten(), likelihood)
    training_iter = 150

    # Find optimal model hyperparameters
    model.train()
    likelihood.train()

    # Use the adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)  # Includes GaussianLikelihood parameters

    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(training_iter):
        # Zero gradients from previous iteration
        optimizer.zero_grad()
        # Output from model
        output = model(X_scaled)
        # Calc loss and backprop gradients
        loss = -mll(output, y_scaled.flatten())
        loss.backward()
        print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
            i + 1, training_iter, loss.item(),
            model.covar_module.base_kernel.lengthscale.item(),
            model.likelihood.noise.item()
        ))
        optimizer.step()

        # print(list(model.named_parameters()))
    
    return model, likelihood

In [23]:
model_likelihood_list = []
for i in range(N // k):
    test_idx = np.arange(k * i, k * (i + 1))
    train_idx = np.delete(np.arange(N), test_idx)
    model, likelihood = train(X_scaled=X_scaled[train_idx], y_scaled=y_scaled[train_idx])
    model_likelihood_list.append((model, likelihood))

Iter 1/150 - Loss: 156.604   lengthscale: 0.693   noise: 0.001
Iter 2/150 - Loss: 145.516   lengthscale: 0.644   noise: 0.001
Iter 3/150 - Loss: 134.931   lengthscale: 0.598   noise: 0.001
Iter 4/150 - Loss: 124.356   lengthscale: 0.554   noise: 0.001
Iter 5/150 - Loss: 113.325   lengthscale: 0.513   noise: 0.001
Iter 6/150 - Loss: 101.264   lengthscale: 0.473   noise: 0.001
Iter 7/150 - Loss: 87.432   lengthscale: 0.437   noise: 0.001
Iter 8/150 - Loss: 71.827   lengthscale: 0.402   noise: 0.001
Iter 9/150 - Loss: 56.279   lengthscale: 0.369   noise: 0.001
Iter 10/150 - Loss: 42.881   lengthscale: 0.339   noise: 0.001
Iter 11/150 - Loss: 31.384   lengthscale: 0.310   noise: 0.001
Iter 12/150 - Loss: 20.657   lengthscale: 0.284   noise: 0.001
Iter 13/150 - Loss: 11.793   lengthscale: 0.260   noise: 0.001
Iter 14/150 - Loss: 6.295   lengthscale: 0.238   noise: 0.001
Iter 15/150 - Loss: 3.587   lengthscale: 0.218   noise: 0.001
Iter 16/150 - Loss: 2.371   lengthscale: 0.201   noise: 0.00

In [24]:
x_grid = torch.linspace(0, 1, 50)
test_x = torch.stack(torch.meshgrid(x_grid, x_grid)).reshape(2, -1).T

In [25]:
for i, (model, likelihood) in enumerate(model_likelihood_list):
    test_idx = np.arange(k * i, k * (i + 1))
    X_test = X_scaled[test_idx]
    
    train_idx = np.delete(np.arange(N), test_idx)

    # Get into evaluation (predictive posterior) mode
    model.eval()
    likelihood.eval()
    # Make predictions by feeding model through likelihood
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(X_test))
    print(y_scaled[test_idx].flatten(), observed_pred.mean)

tensor([2.9865], dtype=torch.float64) tensor([0.4022], dtype=torch.float64)
tensor([0.4140], dtype=torch.float64) tensor([-0.2105], dtype=torch.float64)
tensor([-0.9559], dtype=torch.float64) tensor([0.1189], dtype=torch.float64)
tensor([-0.5599], dtype=torch.float64) tensor([-0.2245], dtype=torch.float64)
tensor([-0.5519], dtype=torch.float64) tensor([0.4763], dtype=torch.float64)
tensor([0.2264], dtype=torch.float64) tensor([0.0825], dtype=torch.float64)
tensor([-0.3873], dtype=torch.float64) tensor([-0.2308], dtype=torch.float64)
tensor([-0.9462], dtype=torch.float64) tensor([-0.0635], dtype=torch.float64)
tensor([-0.9383], dtype=torch.float64) tensor([-0.1664], dtype=torch.float64)
tensor([1.1408], dtype=torch.float64) tensor([0.6868], dtype=torch.float64)
tensor([0.3418], dtype=torch.float64) tensor([-0.0165], dtype=torch.float64)
tensor([-0.9707], dtype=torch.float64) tensor([-0.0398], dtype=torch.float64)
tensor([-0.9218], dtype=torch.float64) tensor([-0.0229], dtype=torch.float

In [26]:
for i, (model, likelihood) in enumerate(model_likelihood_list):
    # Get into evaluation (predictive posterior) mode
    model.eval()
    likelihood.eval()
    # Make predictions by feeding model through likelihood
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        observed_pred = likelihood(model(X_scaled))
    # print(scaler.inverse_transform(y_scaled).flatten())
    print(scaler.inverse_transform(observed_pred.mean[:, None]).flatten())
    print()

[ 8603.90457541  8671.9932542    553.69039371  2900.70232744
  2948.67153397  7561.31069023  3923.34651855   611.04561553
   657.79851694 12981.56441118  8244.82284534   465.69416584
   755.46095248  6012.80557814   887.8186213  12237.82309581
  9309.30618021  5407.60574379  2486.26745324  9406.8200418
  2508.90852203 17860.15912719   489.40588641  2906.9790417
 11764.21339359]

[23920.22550785  4971.03843998   553.6640652   2900.75305757
  2948.97518209  7560.99991803  3923.15359867   610.80452067
   657.62660109 12981.87014294  8244.85690001   465.40275
   755.20431105  6013.18425227   887.95415926 12237.62348033
  9308.86304277  5407.29344952  2485.82513129  9406.7611297
  2508.69801318 17861.7695412    489.00331482  2906.39082449
 11766.48419989]

[23919.60988047  8671.92783656  6924.22351401  2900.72716586
  2948.81454166  7560.97072053  3923.26585733   611.00935733
   657.77933118 12981.51860518  8244.82808167   465.64760538
   755.42329658  6012.86049764   887.9155663  12237.953